In [ ]:
# ================================================================
# DEEP LEARNING LABORATORY
# EXPERIMENT 4 (extension of Exp. 2)
# Data Preprocessing and Neural Network Training
#
# Dataset:
#     Wine Dataset
#
# Topics Covered:
#     1. Data preprocessing
#     2. Train-test splitting
#     3. Feature scaling
#     4. One-hot encoding
#     5. Neural network construction
#     6. Activation function comparison
#     7. Optimizer comparison
###### Part of Exp. 4 ######
#     8. Learning-rate experiment
#     9. Batch-size experiment
#    10. Epoch experiment
#    11. Dropout
#    12. Batch Normalization
#    13. Early Stopping
#    14. L2 Regularization
#    15. Combined regularization
#    16. Training/validation curves
#    17. Final comparison tables
# ================================================================


# ================================================================
# 1. IMPORT LIBRARIES
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.utils import to_categorical

import time


# ================================================================
# 2. REPRODUCIBILITY
# ================================================================

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)


# ================================================================
# 3. LOAD WINE DATASET
# ================================================================

wine = load_wine()

X = wine.data
y = wine.target

print("\n================ DATASET INFORMATION ================")

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("Number of features:", X.shape[1])
print("Number of classes:", len(wine.target_names))

print("\nFeature names:")
print(wine.feature_names)

print("\nClass names:")
print(wine.target_names)


# ================================================================
# 4. DISPLAY FIRST FIVE SAMPLES
# ================================================================

print("\n================ FIRST FIVE SAMPLES ================")

print("First five feature vectors:")
print(X[:5])

print("\nFirst five labels:")
print(y[:5])


# ================================================================
# 5. TRAIN-TEST SPLIT
# ================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n================ TRAIN-TEST SPLIT ================")

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


# ================================================================
# 6. FEATURE SCALING
# ================================================================

scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to test data
X_test_scaled = scaler.transform(X_test)

print("\n================ FEATURE SCALING ================")

print("Original first training sample:")
print(X_train[0])

print("\nScaled first training sample:")
print(X_train_scaled[0])

print("\nScaled training mean (approximately):")
print(np.mean(X_train_scaled, axis=0))


# ================================================================
# 7. ONE-HOT ENCODING
# ================================================================

num_classes = len(wine.target_names)

y_train_encoded = to_categorical(
    y_train,
    num_classes=num_classes
)

y_test_encoded = to_categorical(
    y_test,
    num_classes=num_classes
)

print("\n================ ONE-HOT ENCODING ================")

print("Original labels:")
print(y_train[:10])

print("\nEncoded labels:")
print(y_train_encoded[:10])


# ================================================================
# 8. BASIC MODEL CREATION FUNCTION
# ================================================================

def create_model(
    activation="relu",
    optimizer_name="adam",
    learning_rate=0.001
):

    # ------------------------------------------------------------
    # Select optimizer
    # ------------------------------------------------------------

    if optimizer_name == "adam":

        optimizer = tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        )

    elif optimizer_name == "sgd":

        optimizer = tf.keras.optimizers.SGD(
            learning_rate=learning_rate
        )

    elif optimizer_name == "rmsprop":

        optimizer = tf.keras.optimizers.RMSprop(
            learning_rate=learning_rate
        )

    else:

        raise ValueError(
            "Unsupported optimizer: "
            + str(optimizer_name)
        )

    # ------------------------------------------------------------
    # Create network
    # ------------------------------------------------------------

    model = tf.keras.Sequential([

        tf.keras.Input(shape=(13,)),

        tf.keras.layers.Dense(
            16,
            activation=activation
        ),

        tf.keras.layers.Dense(
            8,
            activation=activation
        ),

        tf.keras.layers.Dense(
            3,
            activation="softmax"
        )
    ])

    # ------------------------------------------------------------
    # Compile
    # ------------------------------------------------------------

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# ================================================================
# 9. FLEXIBLE MODEL FUNCTION
#
# Used for:
#     Dropout
#     Batch Normalization
#     L2 Regularization
# ================================================================

def create_regularized_model(
    activation="relu",
    optimizer_name="adam",
    learning_rate=0.001,
    dropout_rate=0.0,
    use_batch_norm=False,
    l2_rate=0.0
):

    # ------------------------------------------------------------
    # Select optimizer
    # ------------------------------------------------------------

    if optimizer_name == "adam":

        optimizer = tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        )

    elif optimizer_name == "sgd":

        optimizer = tf.keras.optimizers.SGD(
            learning_rate=learning_rate
        )

    elif optimizer_name == "rmsprop":

        optimizer = tf.keras.optimizers.RMSprop(
            learning_rate=learning_rate
        )

    else:

        raise ValueError(
            "Unsupported optimizer: "
            + str(optimizer_name)
        )

    # ------------------------------------------------------------
    # L2 regularizer
    # ------------------------------------------------------------

    if l2_rate > 0:

        regularizer = tf.keras.regularizers.l2(
            l2_rate
        )

    else:

        regularizer = None

    # ------------------------------------------------------------
    # Create model
    # ------------------------------------------------------------

    model = tf.keras.Sequential()

    model.add(
        tf.keras.Input(shape=(13,))
    )

    # ------------------------------------------------------------
    # First hidden layer
    # ------------------------------------------------------------

    model.add(
        tf.keras.layers.Dense(
            16,
            activation=activation,
            kernel_regularizer=regularizer
        )
    )

    if use_batch_norm:

        model.add(
            tf.keras.layers.BatchNormalization()
        )

    if dropout_rate > 0:

        model.add(
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

    # ------------------------------------------------------------
    # Second hidden layer
    # ------------------------------------------------------------

    model.add(
        tf.keras.layers.Dense(
            8,
            activation=activation,
            kernel_regularizer=regularizer
        )
    )

    if use_batch_norm:

        model.add(
            tf.keras.layers.BatchNormalization()
        )

    if dropout_rate > 0:

        model.add(
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

    # ------------------------------------------------------------
    # Output layer
    # ------------------------------------------------------------

    model.add(
        tf.keras.layers.Dense(
            3,
            activation="softmax"
        )
    )

    # ------------------------------------------------------------
    # Compile
    # ------------------------------------------------------------

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# ================================================================
# 10. DISPLAY BASELINE MODEL
# ================================================================

print("\n================ BASELINE MODEL ================")

baseline_model = create_model(
    activation="relu",
    optimizer_name="adam",
    learning_rate=0.001
)

baseline_model.summary()


# ================================================================
# 11. TRAIN BASELINE MODEL
# ================================================================

print("\n================ BASELINE TRAINING ================")

start_time = time.time()

baseline_history = baseline_model.fit(
    X_train_scaled,
    y_train_encoded,
    epochs=30,
    batch_size=16,
    validation_split=0.20,
    verbose=1
)

baseline_training_time = time.time() - start_time


# ================================================================
# 12. EVALUATE BASELINE MODEL
# ================================================================

baseline_train_loss, baseline_train_accuracy = (
    baseline_model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )
)

baseline_test_loss, baseline_test_accuracy = (
    baseline_model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )
)

print("\n================ BASELINE RESULTS ================")

print(
    "Training Accuracy:",
    baseline_train_accuracy
)

print(
    "Test Accuracy:",
    baseline_test_accuracy
)

print(
    "Training Time:",
    round(baseline_training_time, 3),
    "seconds"
)


# ================================================================
# 13. PLOT BASELINE TRAINING CURVES
# ================================================================

plt.figure(figsize=(8, 5))

plt.plot(
    baseline_history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    baseline_history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline: Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    baseline_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    baseline_history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline: Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


# ================================================================
# 14. ACTIVATION FUNCTION EXPERIMENT
# ================================================================

print("\n================================================")
print("ACTIVATION FUNCTION EXPERIMENT")
print("================================================")

activations = [
    "relu",
    "tanh",
    "sigmoid"
]

activation_results = []

for activation in activations:

    print(
        "\nTraining model with activation:",
        activation
    )

    model = create_model(
        activation=activation,
        optimizer_name="adam",
        learning_rate=0.001
    )

    start_time = time.time()

    history = model.fit(
        X_train_scaled,
        y_train_encoded,
        epochs=30,
        batch_size=16,
        validation_split=0.20,
        verbose=0
    )

    training_time = time.time() - start_time

    train_loss, train_accuracy = model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )

    activation_results.append({
        "Activation": activation,
        "Training Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Training Time (s)": training_time
    })


activation_df = pd.DataFrame(
    activation_results
)

print("\nActivation Function Results:")
display(activation_df)


# ================================================================
# 15. OPTIMIZER EXPERIMENT
# ================================================================

print("\n================================================")
print("OPTIMIZER EXPERIMENT")
print("================================================")

optimizers = [
    "sgd",
    "adam",
    "rmsprop"
]

optimizer_results = []

for optimizer_name in optimizers:

    print(
        "\nTraining model with optimizer:",
        optimizer_name
    )

    model = create_model(
        activation="relu",
        optimizer_name=optimizer_name,
        learning_rate=0.001
    )

    start_time = time.time()

    history = model.fit(
        X_train_scaled,
        y_train_encoded,
        epochs=30,
        batch_size=16,
        validation_split=0.20,
        verbose=0
    )

    training_time = time.time() - start_time

    train_loss, train_accuracy = model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )

    optimizer_results.append({
        "Optimizer": optimizer_name,
        "Training Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Training Time (s)": training_time
    })


optimizer_df = pd.DataFrame(
    optimizer_results
)

print("\nOptimizer Results:")
display(optimizer_df)


# ================================================================
# 16. SCALING COMPARISON
# ================================================================

print("\n================================================")
print("SCALING COMPARISON")
print("================================================")


# ------------------------------------------------
# Model WITHOUT scaling
# ------------------------------------------------

model_without_scaling = create_model(
    activation="relu",
    optimizer_name="adam",
    learning_rate=0.001
)

start_time = time.time()

history_without_scaling = model_without_scaling.fit(
    X_train,
    y_train_encoded,
    epochs=30,
    batch_size=16,
    validation_split=0.20,
    verbose=0
)

time_without_scaling = time.time() - start_time

train_loss_ns, train_accuracy_ns = (
    model_without_scaling.evaluate(
        X_train,
        y_train_encoded,
        verbose=0
    )
)

test_loss_ns, test_accuracy_ns = (
    model_without_scaling.evaluate(
        X_test,
        y_test_encoded,
        verbose=0
    )
)


# ------------------------------------------------
# Model WITH scaling
# ------------------------------------------------

model_with_scaling = create_model(
    activation="relu",
    optimizer_name="adam",
    learning_rate=0.001
)

start_time = time.time()

history_with_scaling = model_with_scaling.fit(
    X_train_scaled,
    y_train_encoded,
    epochs=30,
    batch_size=16,
    validation_split=0.20,
    verbose=0
)

time_with_scaling = time.time() - start_time

train_loss_s, train_accuracy_s = (
    model_with_scaling.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )
)

test_loss_s, test_accuracy_s = (
    model_with_scaling.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )
)


scaling_df = pd.DataFrame({

    "Data": [
        "Without Scaling",
        "With Scaling"
    ],

    "Training Accuracy": [
        train_accuracy_ns,
        train_accuracy_s
    ],

    "Test Accuracy": [
        test_accuracy_ns,
        test_accuracy_s
    ],

    "Training Time (s)": [
        time_without_scaling,
        time_with_scaling
    ]
})


print("\nScaling Comparison:")
display(scaling_df)


# ================================================================
# 17. LEARNING RATE EXPERIMENT
# ================================================================

print("\n================================================")
print("LEARNING RATE EXPERIMENT")
print("================================================")

learning_rates = [
    0.1,
    0.01,
    0.001,
    0.0001
]

learning_rate_results = []

for lr in learning_rates:

    print(
        "\nLearning Rate:",
        lr
    )

    model = create_regularized_model(
        activation="relu",
        optimizer_name="adam",
        learning_rate=lr
    )

    start_time = time.time()

    history = model.fit(
        X_train_scaled,
        y_train_encoded,
        epochs=30,
        batch_size=16,
        validation_split=0.20,
        verbose=0
    )

    training_time = time.time() - start_time

    train_loss, train_accuracy = model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )

    learning_rate_results.append({
        "Learning Rate": lr,
        "Training Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Final Validation Loss": history.history["val_loss"][-1],
        "Training Time (s)": training_time
    })


learning_rate_df = pd.DataFrame(
    learning_rate_results
)

print("\nLearning Rate Results:")
display(learning_rate_df)


# ================================================================
# 18. BATCH SIZE EXPERIMENT
# ================================================================

print("\n================================================")
print("BATCH SIZE EXPERIMENT")
print("================================================")

batch_sizes = [
    8,
    16,
    32,
    64
]

batch_results = []

for batch_size in batch_sizes:

    print(
        "\nBatch Size:",
        batch_size
    )

    model = create_regularized_model(
        activation="relu",
        optimizer_name="adam",
        learning_rate=0.001
    )

    start_time = time.time()

    history = model.fit(
        X_train_scaled,
        y_train_encoded,
        epochs=30,
        batch_size=batch_size,
        validation_split=0.20,
        verbose=0
    )

    training_time = time.time() - start_time

    train_loss, train_accuracy = model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )

    batch_results.append({
        "Batch Size": batch_size,
        "Training Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Training Time (s)": training_time
    })


batch_df = pd.DataFrame(
    batch_results
)

print("\nBatch Size Results:")
display(batch_df)


# ================================================================
# 19. EPOCH EXPERIMENT
# ================================================================

print("\n================================================")
print("EPOCH EXPERIMENT")
print("================================================")

epoch_values = [
    10,
    30,
    50,
    100
]

epoch_results = []

for epochs in epoch_values:

    print(
        "\nNumber of Epochs:",
        epochs
    )

    model = create_regularized_model(
        activation="relu",
        optimizer_name="adam",
        learning_rate=0.001
    )

    start_time = time.time()

    history = model.fit(
        X_train_scaled,
        y_train_encoded,
        epochs=epochs,
        batch_size=16,
        validation_split=0.20,
        verbose=0
    )

    training_time = time.time() - start_time

    train_loss, train_accuracy = model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )

    epoch_results.append({
        "Epochs": epochs,
        "Training Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Final Validation Accuracy":
            history.history["val_accuracy"][-1],
        "Training Time (s)": training_time
    })


epoch_df = pd.DataFrame(
    epoch_results
)

print("\nEpoch Results:")
display(epoch_df)


# ================================================================
# 20. DROPOUT EXPERIMENT
# ================================================================

print("\n================================================")
print("DROPOUT EXPERIMENT")
print("================================================")

dropout_rates = [
    0.0,
    0.2,
    0.5
]

dropout_results = []

for rate in dropout_rates:

    print(
        "\nDropout Rate:",
        rate
    )

    model = create_regularized_model(
        activation="relu",
        optimizer_name="adam",
        learning_rate=0.001,
        dropout_rate=rate
    )

    start_time = time.time()

    history = model.fit(
        X_train_scaled,
        y_train_encoded,
        epochs=50,
        batch_size=16,
        validation_split=0.20,
        verbose=0
    )

    training_time = time.time() - start_time

    train_loss, train_accuracy = model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )

    dropout_results.append({
        "Dropout Rate": rate,
        "Training Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Final Validation Accuracy":
            history.history["val_accuracy"][-1],
        "Training Time (s)": training_time
    })


dropout_df = pd.DataFrame(
    dropout_results
)

print("\nDropout Results:")
display(dropout_df)


# ================================================================
# 21. BATCH NORMALIZATION EXPERIMENT
# ================================================================

print("\n================================================")
print("BATCH NORMALIZATION EXPERIMENT")
print("================================================")

batch_norm_results = []

for use_bn in [False, True]:

    print(
        "\nBatch Normalization:",
        use_bn
    )

    model = create_regularized_model(
        activation="relu",
        optimizer_name="adam",
        learning_rate=0.001,
        use_batch_norm=use_bn
    )

    start_time = time.time()

    history = model.fit(
        X_train_scaled,
        y_train_encoded,
        epochs=50,
        batch_size=16,
        validation_split=0.20,
        verbose=0
    )

    training_time = time.time() - start_time

    train_loss, train_accuracy = model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )

    batch_norm_results.append({
        "Batch Normalization":
            "Yes" if use_bn else "No",
        "Training Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Final Validation Accuracy":
            history.history["val_accuracy"][-1],
        "Training Time (s)": training_time
    })


batch_norm_df = pd.DataFrame(
    batch_norm_results
)

print("\nBatch Normalization Results:")
display(batch_norm_df)


# ================================================================
# 22. EARLY STOPPING EXPERIMENT
# ================================================================

print("\n================================================")
print("EARLY STOPPING EXPERIMENT")
print("================================================")

early_stopping = tf.keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True
)


early_model = create_regularized_model(
    activation="relu",
    optimizer_name="adam",
    learning_rate=0.001
)


start_time = time.time()

early_history = early_model.fit(

    X_train_scaled,

    y_train_encoded,

    epochs=100,

    batch_size=16,

    validation_split=0.20,

    callbacks=[early_stopping],

    verbose=1
)


early_training_time = time.time() - start_time


early_train_loss, early_train_accuracy = (
    early_model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )
)


early_test_loss, early_test_accuracy = (
    early_model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )
)


early_epochs_completed = len(
    early_history.history["loss"]
)


print("\nEarly Stopping Results:")

print(
    "Training Accuracy:",
    early_train_accuracy
)

print(
    "Test Accuracy:",
    early_test_accuracy
)

print(
    "Epochs Completed:",
    early_epochs_completed
)

print(
    "Training Time:",
    round(
        early_training_time,
        3
    ),
    "seconds"
)


# ================================================================
# 23. EARLY STOPPING CURVES
# ================================================================

plt.figure(figsize=(8, 5))

plt.plot(
    early_history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    early_history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Early Stopping: Accuracy")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    early_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    early_history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Early Stopping: Loss")
plt.legend()
plt.grid(True)
plt.show()


# ================================================================
# 24. L2 REGULARIZATION EXPERIMENT
# ================================================================

print("\n================================================")
print("L2 REGULARIZATION EXPERIMENT")
print("================================================")

l2_values = [
    0.0,
    0.0001,
    0.001,
    0.01
]

l2_results = []

for l2_value in l2_values:

    print(
        "\nL2 Rate:",
        l2_value
    )

    model = create_regularized_model(
        activation="relu",
        optimizer_name="adam",
        learning_rate=0.001,
        l2_rate=l2_value
    )

    start_time = time.time()

    history = model.fit(
        X_train_scaled,
        y_train_encoded,
        epochs=50,
        batch_size=16,
        validation_split=0.20,
        verbose=0
    )

    training_time = time.time() - start_time

    train_loss, train_accuracy = model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )

    l2_results.append({
        "L2 Rate": l2_value,
        "Training Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Final Validation Accuracy":
            history.history["val_accuracy"][-1],
        "Training Time (s)": training_time
    })


l2_df = pd.DataFrame(
    l2_results
)

print("\nL2 Regularization Results:")
display(l2_df)


# ================================================================
# 25. COMBINED REGULARIZATION EXPERIMENT
# ================================================================

print("\n================================================")
print("COMBINED REGULARIZATION EXPERIMENT")
print("================================================")

combined_model = create_regularized_model(

    activation="relu",

    optimizer_name="adam",

    learning_rate=0.001,

    dropout_rate=0.2,

    use_batch_norm=True,

    l2_rate=0.0001
)


combined_early_stopping = (
    tf.keras.callbacks.EarlyStopping(

        monitor="val_loss",

        patience=10,

        restore_best_weights=True
    )
)


start_time = time.time()

combined_history = combined_model.fit(

    X_train_scaled,

    y_train_encoded,

    epochs=100,

    batch_size=16,

    validation_split=0.20,

    callbacks=[combined_early_stopping],

    verbose=1
)


combined_training_time = (
    time.time() - start_time
)


# ================================================================
# 26. EVALUATE COMBINED MODEL
# ================================================================

combined_train_loss, combined_train_accuracy = (
    combined_model.evaluate(
        X_train_scaled,
        y_train_encoded,
        verbose=0
    )
)


combined_test_loss, combined_test_accuracy = (
    combined_model.evaluate(
        X_test_scaled,
        y_test_encoded,
        verbose=0
    )
)


combined_epochs = len(
    combined_history.history["loss"]
)


print("\n================ COMBINED MODEL RESULTS ================")

print(
    "Training Accuracy:",
    combined_train_accuracy
)

print(
    "Test Accuracy:",
    combined_test_accuracy
)

print(
    "Epochs Completed:",
    combined_epochs
)

print(
    "Training Time:",
    round(
        combined_training_time,
        3
    ),
    "seconds"
)


# ================================================================
# 27. COMBINED MODEL TRAINING CURVES
# ================================================================

plt.figure(figsize=(8, 5))

plt.plot(
    combined_history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    combined_history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title(
    "Combined Regularization: Accuracy"
)

plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    combined_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    combined_history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Combined Regularization: Loss"
)

plt.legend()
plt.grid(True)
plt.show()


# ================================================================
# 28. TRAINING-TEST ACCURACY GAP
# ================================================================

baseline_gap = (
    baseline_train_accuracy
    - baseline_test_accuracy
)

combined_gap = (
    combined_train_accuracy
    - combined_test_accuracy
)


gap_df = pd.DataFrame({

    "Model": [
        "Baseline",
        "Combined Regularization"
    ],

    "Training Accuracy": [
        baseline_train_accuracy,
        combined_train_accuracy
    ],

    "Test Accuracy": [
        baseline_test_accuracy,
        combined_test_accuracy
    ],

    "Training-Test Accuracy Gap": [
        baseline_gap,
        combined_gap
    ]
})


print("\n================================================")
print("GENERALIZATION GAP")
print("================================================")

display(gap_df)


# ================================================================
# 29. FINAL REGULARIZATION COMPARISON
# ================================================================

regularization_comparison = pd.DataFrame({

    "Method": [
        "Baseline",
        "Dropout",
        "Batch Normalization",
        "Early Stopping",
        "L2 Regularization",
        "Combined"
    ],

    "Training Accuracy": [

        baseline_train_accuracy,

        dropout_df[
            "Training Accuracy"
        ].iloc[1],

        batch_norm_df[
            "Training Accuracy"
        ].iloc[1],

        early_train_accuracy,

        l2_df[
            "Training Accuracy"
        ].iloc[1],

        combined_train_accuracy
    ],

    "Test Accuracy": [

        baseline_test_accuracy,

        dropout_df[
            "Test Accuracy"
        ].iloc[1],

        batch_norm_df[
            "Test Accuracy"
        ].iloc[1],

        early_test_accuracy,

        l2_df[
            "Test Accuracy"
        ].iloc[1],

        combined_test_accuracy
    ],

    "Training-Test Gap": [

        baseline_train_accuracy
        - baseline_test_accuracy,

        dropout_df[
            "Training Accuracy"
        ].iloc[1]
        -
        dropout_df[
            "Test Accuracy"
        ].iloc[1],

        batch_norm_df[
            "Training Accuracy"
        ].iloc[1]
        -
        batch_norm_df[
            "Test Accuracy"
        ].iloc[1],

        early_train_accuracy
        - early_test_accuracy,

        l2_df[
            "Training Accuracy"
        ].iloc[1]
        -
        l2_df[
            "Test Accuracy"
        ].iloc[1],

        combined_train_accuracy
        - combined_test_accuracy
    ]
})


print("\n================================================")
print("FINAL REGULARIZATION COMPARISON")
print("================================================")

display(
    regularization_comparison
)


# ================================================================
# 30. PLOT REGULARIZATION COMPARISON
# ================================================================

plt.figure(figsize=(10, 5))

plt.bar(
    regularization_comparison["Method"],
    regularization_comparison["Test Accuracy"]
)

plt.xlabel("Method")
plt.ylabel("Test Accuracy")

plt.title(
    "Comparison of Training and Regularization Methods"
)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.grid(
    axis="y"
)

plt.tight_layout()

plt.show()


# ================================================================
# 31. TRAINING VS TEST ACCURACY
# ================================================================

x = np.arange(
    len(
        regularization_comparison["Method"]
    )
)

width = 0.35

plt.figure(figsize=(11, 5))

plt.bar(
    x - width / 2,
    regularization_comparison[
        "Training Accuracy"
    ],
    width,
    label="Training Accuracy"
)

plt.bar(
    x + width / 2,
    regularization_comparison[
        "Test Accuracy"
    ],
    width,
    label="Test Accuracy"
)

plt.xlabel("Method")
plt.ylabel("Accuracy")

plt.title(
    "Training Accuracy vs Test Accuracy"
)

plt.xticks(
    x,
    regularization_comparison["Method"],
    rotation=30,
    ha="right"
)

plt.legend()

plt.grid(
    axis="y"
)

plt.tight_layout()

plt.show()


# ================================================================
# 32. FINAL COMPLETE EXPERIMENT SUMMARY
# ================================================================

print("\n")
print("============================================================")
print("FINAL EXPERIMENT SUMMARY")
print("============================================================")


print("\n1. Dataset")
print("--------------------------------------------")
print("Dataset: Wine")
print("Samples:", X.shape[0])
print("Features:", X.shape[1])
print("Classes:", num_classes)


print("\n2. Baseline")
print("--------------------------------------------")
print(
    "Training Accuracy:",
    round(baseline_train_accuracy, 4)
)
print(
    "Test Accuracy:",
    round(baseline_test_accuracy, 4)
)
print(
    "Training Time:",
    round(baseline_training_time, 3),
    "seconds"
)


print("\n3. Activation Functions")
print("--------------------------------------------")
display(activation_df)


print("\n4. Optimizers")
print("--------------------------------------------")
display(optimizer_df)


print("\n5. Feature Scaling")
print("--------------------------------------------")
display(scaling_df)


print("\n6. Learning Rate")
print("--------------------------------------------")
display(learning_rate_df)


print("\n7. Batch Size")
print("--------------------------------------------")
display(batch_df)


print("\n8. Number of Epochs")
print("--------------------------------------------")
display(epoch_df)


print("\n9. Dropout")
print("--------------------------------------------")
display(dropout_df)


print("\n10. Batch Normalization")
print("--------------------------------------------")
display(batch_norm_df)


print("\n11. Early Stopping")
print("--------------------------------------------")

print(
    "Training Accuracy:",
    round(early_train_accuracy, 4)
)

print(
    "Test Accuracy:",
    round(early_test_accuracy, 4)
)

print(
    "Epochs Completed:",
    early_epochs_completed
)


print("\n12. L2 Regularization")
print("--------------------------------------------")
display(l2_df)


print("\n13. Combined Regularization")
print("--------------------------------------------")

print(
    "Training Accuracy:",
    round(combined_train_accuracy, 4)
)

print(
    "Test Accuracy:",
    round(combined_test_accuracy, 4)
)

print(
    "Epochs Completed:",
    combined_epochs
)


print("\n============================================================")
print("END OF EXPERIMENT 2")
print("============================================================")